In [4]:
from datasets import load_dataset

print("Downloading and loading the IMDB dataset...")
# Updated with the strict namespace format Hugging Face now requires
dataset = load_dataset("stanfordnlp/imdb")

# Extract the text and labels into standard Python lists
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

print(f"Successfully loaded {len(train_texts)} training reviews and {len(test_texts)} test reviews.")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 108404.09 examples/s]


Successfully loaded 25000 training reviews and 25000 test reviews.


In [6]:
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 3.9 MB/s eta 0:00:04
     ----- ---------------------------------- 1.8/12.8 MB 3.9 MB/s eta 0:00:03
     ------- -------------------------------- 2.4/12.8 MB 3.4 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 3.5 MB/s eta 0:00:03
     ------------- -------------------------- 4.2/12.8 MB 3.6 MB/s eta 0:00:03
     ---------------- ----------------------- 5.2/12.8 MB 3.8 MB/s eta 0:00:02
     ------------------ --------------------- 5.8/12.8 MB 3.9 MB/s eta 0:00:02
     --------------------- ------------------ 6.8/12.8 MB 3.8 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 3.7 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 3.7 MB/s 


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\attil\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import spacy

# Load the small English model you installed earlier
nlp = spacy.load("en_core_web_sm")

def preprocess_reviews(reviews):
    clean_reviews = []
    
    # nlp.pipe is much faster than a standard for-loop for large datasets
    # Use enumerate to keep track of the loop index (i)
    for i, doc in enumerate(nlp.pipe(reviews, batch_size=500)):
        # Print a progress update every 5,000 reviews
        if i % 5000 == 0 and i > 0:
            print(f"Processed {i} reviews...")
        clean_tokens = []
        for token in doc:
            # Filter out stop words, punctuation, and white spaces
            if not token.is_stop and not token.is_punct and not token.is_space:
                # Apply lemmatization to get the base form and convert to lowercase
                clean_tokens.append(token.lemma_.lower())
                
        # Rejoin the clean tokens into a single string for the model
        clean_reviews.append(" ".join(clean_tokens))
        
    return clean_reviews

# Pass the raw text arrays you created in the previous step
print("Processing training data...")
clean_train_texts = preprocess_reviews(train_texts)

print("Processing test data...")
clean_test_texts = preprocess_reviews(test_texts)

Processing training data...
Processed 5000 reviews...
Processed 10000 reviews...
Processed 15000 reviews...
Processed 20000 reviews...
Processing test data...
Processed 5000 reviews...
Processed 10000 reviews...
Processed 15000 reviews...
Processed 20000 reviews...


In [9]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\attil\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
from tensorflow.keras.layers import TextVectorization, Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

# 1. Feature Encoding (Vectorization)
vocab_size = 10000  # Only keep the top 10,000 most frequent words
max_length = 200    # Cap reviews at 200 words

# Initialize the vectorization layer
vectorize_layer = TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=max_length
)

# Build the vocabulary dictionary based on your clean training data
print("Building vocabulary...")
vectorize_layer.adapt(clean_train_texts)

# 2. Building the Deep Learning Model
print("Building the neural network...")
model = Sequential([
    # Step 1: Input the vectorized text
    vectorize_layer,
    
    # Step 2: Embed words into a semantic vector space
    Embedding(input_dim=vocab_size, output_dim=64),
    
    # Step 3: RNN layer to process the sequence of words
    LSTM(64),
    
    # Step 4: Output layer for binary classification 
    Dense(1, activation='sigmoid')
])

# Compile the model with metrics to evaluate accuracy
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Building vocabulary...
Building the neural network...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils import shuffle

print("Shuffling and converting data to TensorFlow Tensors...")

# 1. Shuffle the raw training text and labels in unison
shuffled_train_texts, shuffled_train_labels = shuffle(clean_train_texts, train_labels, random_state=42)

# 2. Convert to TensorFlow Tensors
X_train_tensor = tf.convert_to_tensor(shuffled_train_texts, dtype=tf.string)
X_test_tensor = tf.convert_to_tensor(clean_test_texts, dtype=tf.string)

y_train_tensor = tf.convert_to_tensor(shuffled_train_labels, dtype=tf.int32)
y_test_tensor = tf.convert_to_tensor(test_labels, dtype=tf.int32)

# ---------------------------------------------------------
# Scenario A: Progressive Testing on a Small Subset
# ---------------------------------------------------------
print("--- Training on a Small Subset (2,000 reviews) ---")
# Slice the new shuffled Tensors
small_subset_texts = X_train_tensor[:2000]
small_subset_labels = y_train_tensor[:2000]

# Train the model
model.fit(
    small_subset_texts, 
    small_subset_labels, 
    epochs=3, 
    batch_size=32, 
    validation_split=0.2
)

# ---------------------------------------------------------
# Scenario B: Training on the Full Dataset
# ---------------------------------------------------------
print("\n--- Training on the Full Dataset (25,000 reviews) ---")
history = model.fit(
    X_train_tensor, 
    y_train_tensor, 
    epochs=5, 
    batch_size=64, 
    validation_split=0.2
)

# ---------------------------------------------------------
# Evaluation on the Test Dataset
# ---------------------------------------------------------
print("\n--- Evaluating on Test Data ---")
predicted_probabilities = model.predict(X_test_tensor)
predicted_labels = (predicted_probabilities > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test_tensor, predicted_labels, target_names=['Negative', 'Positive']))

print("Confusion Matrix:")
print(confusion_matrix(y_test_tensor, predicted_labels))

print(f"\nROC AUC Score: {roc_auc_score(y_test_tensor, predicted_probabilities):.4f}")

Shuffling and converting data to TensorFlow Tensors...
--- Training on a Small Subset (2,000 reviews) ---
Epoch 1/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.8256 - loss: 0.4034 - val_accuracy: 0.8325 - val_loss: 0.4171
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.9038 - loss: 0.2752 - val_accuracy: 0.8425 - val_loss: 0.3998
Epoch 3/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.9394 - loss: 0.1934 - val_accuracy: 0.8575 - val_loss: 0.4213

--- Training on the Full Dataset (25,000 reviews) ---
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 121ms/step - accuracy: 0.8832 - loss: 0.2996 - val_accuracy: 0.8878 - val_loss: 0.2837
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 115ms/step - accuracy: 0.9269 - loss: 0.2083 - val_accuracy: 0.8886 - val_loss: 0.2807
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.9524 - loss: 0.1530 - val_accuracy: 0.8832 - val_loss: 0.3147
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 118ms/step - accuracy: 0.968